# Stage 1 Query-Only Retrieval Selection — Facial Skincare

This notebook evaluates four non-personalized Global Review retrieval methods under one shared exact top-K contract:

- Dense BERT
- BM25
- Dense-BM25 Hybrid
- Graph-Hybrid

All methods use the same target-review-only synthetic queries. Item-side retrieval evidence combines catalog metadata, product-functional facets, and frozen historical review-derived item signals produced before the training cutoff. Graph-Hybrid uses separate metadata functional-facet and historical review-derived graph channels in addition to the shared dense and sparse item representations. Brand is retained in the catalog representation, item-facet table, and item-brand graph edges, but synthetic queries do not contain brand terms and the brand graph channel is not scored directly by query matching.

The deterministic lexicographic selection rule ranks the four methods at the fixed candidate depth of 1,000. The selected query-only winner is recorded in a machine-readable contract containing its method identity, exact candidate budget, and reproducibility lineage.

No user-specific prior information is used in this notebook. Candidate exports retain the catalog brand facet so downstream personalized retrieval and reranking can use brand preference without adding brand to the synthetic query.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q -U pyarrow sentence-transformers faiss-cpu rank-bm25 tqdm

In [ ]:
# =========================================================
# Config
# =========================================================
from pathlib import Path
from collections import defaultdict
import heapq
import json
import math
import re
import time

import faiss
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

CATEGORY_ID = "face"
CATEGORY_LABEL = "Facial Skincare"
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

QUERY_CACHE_PATH = PROJECT_ROOT / "outputs/query_cache/face_queries.parquet"
QUERY_CONTRACT_PATH = PROJECT_ROOT / "outputs/query_summary/face_queries_config.json"
ITEM_DOCS_PATH = PROJECT_ROOT / "data/processed/items/face_item_docs.parquet"
ITEM_FACETS_PATH = PROJECT_ROOT / "data/processed/items/face_items_facets.parquet"
GRAPH_EDGES_PATH = PROJECT_ROOT / "data/processed/items/face_item_graph_edges.parquet"
SAMPLED_USERS_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sample.parquet"
FINAL_SAMPLING_POOL_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_final_sampling_pool.parquet"
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROJECT_ROOT / "data/processed/items/retrieval_artifact_manifest_face.json"

ITEM_ID_COLUMN = "parent_asin"
DENSE_TEXT_COLUMN = "dense_text"
SPARSE_TEXT_COLUMN = "sparse_text"
BRAND_TEXT_COLUMN = "brand_facet_text"
FACET_ITEM_ID_COLUMN = "parent_asin"
FACET_VALUE_COLUMN = "facet_value_norm"
EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
BRAND_QUERY_MATCHING_ENABLED = False
BRAND_IN_FUNCTIONAL_GRAPH = False

EXPECTED_ROWS = None
EXPECTED_REGIME_COUNTS = None
REGIME_ORDER = ["cold", "weak", "moderate", "strong"]
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
EXPECTED_QUERY_VARIANT = "medium_heavy_review_only_dspy_linguistic"

ACTIVE_QUERY_COLUMN = "query"
COMPATIBILITY_QUERY_ALIAS = None
KNOWN_QUERY_ALIAS_COLUMNS = {"query_C"}
CLEAN_QUERY_COLUMN = "query_clean"
QUERY_METHOD_LABEL = "query"
QUERY_STRUCTURE_COLUMNS = [
    "query_generic_anchor_rate",
    "query_specific_facet_cue_rate",
    "query_specific_facet_family_count",
]
QUERY_PASSTHROUGH_COLUMNS = ["sampling_bracket"]
OPTIONAL_QUERY_AUDIT_COLUMNS = [
    CLEAN_QUERY_COLUMN,
    "query_clean_is_active",
    "query_generic_anchor_terms",
    "query_generic_utility_terms",
    "query_generation_status",
    "replacement_case_used",
    "replacement_source_case_id",
]
ADDITIONAL_GROUP_COLUMNS = ["sampling_bracket"]

METHOD_KEYS = [
    "dense_bert",
    "bm25",
    "hybrid_dense_bm25",
    "graph_hybrid",
]
METHOD_DISPLAY_NAMES = {
    "dense_bert": "Dense BERT",
    "bm25": "BM25",
    "hybrid_dense_bm25": "Dense-BM25 Hybrid",
    "graph_hybrid": "Graph-Hybrid",
}

RRF_K = 60
HYBRID_WEIGHTS = {"dense": 1.00, "bm25": 1.00}
GRAPH_HYBRID_WEIGHTS = {
    "dense": 0.90,
    "bm25": 1.00,
    "metadata_graph": 0.50,
    "review_reputation_graph": 0.35,
}

EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]
MAX_RETRIEVAL_K = 1000
SELECTION_K = MAX_RETRIEVAL_K
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ITEM_EMBEDDING_BATCH_SIZE = 128
QUERY_EMBEDDING_BATCH_SIZE = 128
CANDIDATE_EXPORT_QUERY_BATCH_SIZE = 25

RUN_SMOKE_TEST = False
SMOKE_TEST_N = 20
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TOKEN_EQUIVALENCE = {
    "sensitive": "sensitivity", "sensitivity": "sensitivity",
    "oily": "oiliness", "oiliness": "oiliness",
    "dry": "dryness", "dryness": "dryness",
    "wrinkle": "wrinkles", "wrinkles": "wrinkles",
    "hydrating": "hydration", "hydrate": "hydration",
    "hydration": "hydration", "hydrated": "hydration",
    "moisturizer": "moisturizer", "moisturizers": "moisturizer",
    "moisturizing": "moisturizer", "moisturize": "moisturizer",
    "cream": "moisturizer", "creams": "moisturizer",
    "lotion": "moisturizer", "lotions": "moisturizer",
    "serum": "serum", "serums": "serum",
    "mask": "mask", "masks": "mask",
    "cleanser": "cleanser", "cleansers": "cleanser",
    "cleansing": "cleanser", "wash": "cleanser",
    "toner": "toner", "toners": "toner",
    "treatment": "treatment", "treatments": "treatment",
    "acne": "acne", "blemish": "acne", "blemishes": "acne",
    "breakout": "acne", "breakouts": "acne",
    "pore": "pores", "pores": "pores",
    "brightening": "brightening", "brighten": "brightening",
    "brightness": "brightening",
    "hyperpigmentation": "dark_spots", "pigmentation": "dark_spots",
    "spots": "dark_spots", "spot": "dark_spots", "dark": "dark",
}
LINGUISTIC_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "in", "into", "is", "it", "of", "on", "or", "that", "the", "this",
    "to", "with", "without", "you", "your",
}
GENERIC_ANCHOR_TOKENS = {
    "product", "products", "item", "items", "solution", "solutions",
    "face", "facial", "skin", "skincare", "care", "routine", "support",
}
GENERIC_UTILITY_TOKENS = {
    "support", "wellness", "natural", "formula", "blend", "complex",
    "routine", "care", "product", "products", "solution", "solutions",
}
GRAPH_STOPWORDS = (
    LINGUISTIC_STOPWORDS
    | GENERIC_ANCHOR_TOKENS
    | GENERIC_UTILITY_TOKENS
    | {"query", "using", "use"}
)
GRAPH_SHORT_TOKENS = {"b5", "e", "c"}
MIN_GRAPH_TOKEN_LENGTH = 3
GRAPH_MAX_PHRASE_TOKENS = 5
GRAPH_PHRASE_BONUS = 1.25
FORBIDDEN_FACET_SOURCE_TERMS = ("identifier", "count", "policy", "diagnostic")

OUTPUT_DIR = PROJECT_ROOT / "outputs/stage1_query_retrieval_selection"
ACTIVE_OUTPUT_DIR = OUTPUT_DIR / "smoke_test" if RUN_SMOKE_TEST else OUTPUT_DIR
ACTIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_OVERALL_PATH = ACTIVE_OUTPUT_DIR / "stage1_results_overall_face.csv"
RESULTS_BY_POOL_DEPTH_PATH = ACTIVE_OUTPUT_DIR / "stage1_results_by_pool_depth_face.csv"
RESULTS_BY_REGIME_PATH = ACTIVE_OUTPUT_DIR / "stage1_results_by_regime_face.csv"
RESULTS_BY_REGIME_POOL_DEPTH_PATH = ACTIVE_OUTPUT_DIR / "stage1_results_by_regime_pool_depth_face.csv"
RESULTS_BY_QUERY_STRUCTURE_PATH = ACTIVE_OUTPUT_DIR / "stage1_results_by_query_structure_face.csv"
PER_QUERY_METRICS_PATH = ACTIVE_OUTPUT_DIR / "stage1_per_query_metrics_face.parquet"
QUERY_DIAGNOSTICS_PATH = ACTIVE_OUTPUT_DIR / "stage1_query_diagnostics_face.parquet"
EVALUATION_QUERIES_PATH = ACTIVE_OUTPUT_DIR / "stage1_evaluation_queries_face.parquet"
RUNTIME_PATH = ACTIVE_OUTPUT_DIR / "stage1_runtime_top1000_face.csv"
RUNTIME_COMPONENTS_PATH = ACTIVE_OUTPUT_DIR / "stage1_runtime_components_face.csv"
COMPONENT_COVERAGE_PATH = ACTIVE_OUTPUT_DIR / "stage1_component_query_coverage_face.csv"
FACET_FILTER_AUDIT_PATH = ACTIVE_OUTPUT_DIR / "stage1_facet_filter_audit_face.csv"
METHOD_SELECTION_PATH = ACTIVE_OUTPUT_DIR / "stage1_method_selection_face.csv"
RUN_MANIFEST_PATH = ACTIVE_OUTPUT_DIR / "stage1_run_manifest_face.json"

WINNER_MANIFEST_PATH = ACTIVE_OUTPUT_DIR / f"stage1_query_only_winner_{CATEGORY_ID}.json"
WINNER_CONTRACT_VERSION = "stage1_query_only_winner_v1"
WINNER_METHOD_OVERRIDE = None
ADDITIONAL_GROUP_OUTPUT_PATHS = {
    "sampling_bracket": ACTIVE_OUTPUT_DIR / "stage1_results_by_sampling_bracket_face.csv",
}

CANDIDATE_DIR = ACTIVE_OUTPUT_DIR / "query_candidate_cache" if RUN_SMOKE_TEST else OUTPUT_DIR / "query_candidate_cache"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
DENSE_CANDIDATES_PATH = CANDIDATE_DIR / "query_dense_bert_core_top1000_face.parquet"
BM25_CANDIDATES_PATH = CANDIDATE_DIR / "query_bm25_core_top1000_face.parquet"
HYBRID_CANDIDATES_PATH = CANDIDATE_DIR / "query_dense_bm25_hybrid_core_top1000_face.parquet"
GRAPH_HYBRID_CANDIDATES_PATH = CANDIDATE_DIR / "query_C_graph_hybrid_top1000_face.parquet"
METHOD_CANDIDATE_PATHS = {
    "dense_bert": DENSE_CANDIDATES_PATH,
    "bm25": BM25_CANDIDATES_PATH,
    "hybrid_dense_bm25": HYBRID_CANDIDATES_PATH,
    "graph_hybrid": GRAPH_HYBRID_CANDIDATES_PATH,
}

print("Input:", QUERY_CACHE_PATH)
print("Input:", ITEM_DOCS_PATH)
print("Input:", ITEM_FACETS_PATH)
print("Input:", GRAPH_EDGES_PATH)
print("Output:", ACTIVE_OUTPUT_DIR)
print("Winner contract:", WINNER_MANIFEST_PATH)


In [ ]:
# =========================================================
# Shared Functions
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def canonical_tokens(value):
    text = normalize_space(value).lower().replace("_", " ").replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[a-z0-9']+", text)
    return [TOKEN_EQUIVALENCE.get(token, token) for token in tokens]


def canonical_text(value):
    return " ".join(canonical_tokens(value))


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} is missing required columns: {missing}")


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def validate_query_contract(contract):
    if not isinstance(contract, dict) or not contract:
        raise RuntimeError("Notebook 06 query contract must be a non-empty JSON object.")
    if contract.get("active_query_column") != ACTIVE_QUERY_COLUMN:
        raise RuntimeError("Notebook 06 active_query_column must be query.")
    if contract.get("evidence_scope") != "target_review_safe_signals_only":
        raise RuntimeError("Notebook 06 evidence_scope must be target_review_safe_signals_only.")
    if contract.get("query_evidence_source") != "target_review_safe_signals_only":
        raise RuntimeError("Notebook 06 query_evidence_source mismatch.")
    if contract.get("query_variant") != EXPECTED_QUERY_VARIANT:
        raise RuntimeError("Notebook 06 query_variant mismatch.")
    contract_target_per_regime = int(contract.get("target_per_regime", -1))
    if contract_target_per_regime <= 0:
        raise RuntimeError("Notebook 06 target_per_regime must be positive.")
    if list(contract.get("regime_order", [])) != REGIME_ORDER:
        raise RuntimeError("Notebook 06 regime_order mismatch.")
    expected_false = [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "query_clean_is_active",
    ]
    bad_flags = [key for key in expected_false if contract.get(key) is not False]
    if bad_flags:
        raise RuntimeError(f"Notebook 06 contract flags must be false: {bad_flags}")
    if contract.get("user_prior_columns_loaded") not in (None, []):
        raise RuntimeError("Notebook 06 must not load user-prior columns.")
    if contract.get("user_prior_columns_exported") not in (None, []):
        raise RuntimeError("Notebook 06 must not export user-prior columns.")
    aliases = contract.get("compatibility_query_aliases", {})
    if COMPATIBILITY_QUERY_ALIAS is None:
        if aliases not in ({}, None):
            raise RuntimeError("Unexpected Notebook 06 compatibility query alias.")
    elif aliases != {COMPATIBILITY_QUERY_ALIAS: ACTIVE_QUERY_COLUMN}:
        raise RuntimeError("Notebook 06 compatibility query alias mismatch.")


def validate_retrieval_manifest(manifest):
    if not isinstance(manifest, dict) or not manifest:
        raise RuntimeError("Notebook 04 retrieval manifest must be a non-empty JSON object.")
    expected = {
        "evidence_scope": EVIDENCE_SCOPE,
        "historical_review_reputation_enabled": True,
        "review_reputation_graph_enabled": True,
        "brand_graph_enabled": True,
        "brand_in_retrieval_text": True,
        "brand_in_profile_source_text": True,
        "brand_in_synthetic_query": False,
        "raw_review_text_exported": False,
        "dense_source": DENSE_TEXT_COLUMN,
        "sparse_source": SPARSE_TEXT_COLUMN,
        "item_facet_export_option": "B_all_rows_with_reliable_flags",
    }
    mismatches = {
        key: manifest.get(key)
        for key, expected_value in expected.items()
        if manifest.get(key) != expected_value
    }
    if mismatches:
        raise RuntimeError(f"Notebook 04 retrieval manifest mismatch: {mismatches}")
    brand_in_functional_graph = manifest.get(
        "brand_in_functional_graph",
        BRAND_IN_FUNCTIONAL_GRAPH,
    )
    if brand_in_functional_graph is not BRAND_IN_FUNCTIONAL_GRAPH:
        raise RuntimeError(
            "Notebook 04 manifest must declare or imply brand_in_functional_graph=False."
        )
    if not normalize_space(manifest.get("brand_graph_mask")):
        raise RuntimeError("Notebook 04 manifest must document the brand graph mask.")
    if not normalize_space(manifest.get("global_review_graph_mask")):
        raise RuntimeError("Notebook 04 manifest must document the Global Review graph mask.")


def validate_false_columns(frame, columns, frame_name):
    for column in columns:
        if boolean_series(frame[column]).any():
            raise RuntimeError(f"{frame_name}.{column} must be false for every row.")


GRAPH_FACET_FLAG_COLUMNS = [
    "is_product_functional_facet",
    "is_query_safe",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]


def normalize_facet_flags(frame):
    normalized = frame.copy()
    for column in GRAPH_FACET_FLAG_COLUMNS:
        normalized[column] = boolean_series(normalized[column])
    return normalized


def explicit_metadata_facet_mask(facets):
    return (
        facets["is_product_functional_facet"]
        & facets["is_query_safe"]
        & ~facets["is_brand"]
        & ~facets["is_review_derived"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & facets["is_metadata_facet_source"]
        & ~facets["is_disallowed_nonfacet_source"]
    )


def explicit_review_facet_mask(facets):
    return (
        facets["is_review_derived"]
        & ~facets["is_brand"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & ~facets["is_disallowed_nonfacet_source"]
    )


def explicit_brand_facet_mask(facets):
    return (
        facets["facet_role"].eq("brand")
        & facets["is_brand"]
        & ~facets["is_review_derived"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & facets["is_metadata_facet_source"]
        & ~facets["is_disallowed_nonfacet_source"]
    )

def explicit_global_review_facet_mask(facets):
    return (
        explicit_metadata_facet_mask(facets)
        | explicit_brand_facet_mask(facets)
        | explicit_review_facet_mask(facets)
    )


def audit_global_review_facet_flags(raw_facet_flags):
    required = [FACET_ITEM_ID_COLUMN, "source_column", "facet_role", *GRAPH_FACET_FLAG_COLUMNS]
    require_columns(raw_facet_flags, required, "item facet flags")
    facets = normalize_facet_flags(raw_facet_flags[required])

    metadata_mask = explicit_metadata_facet_mask(facets)
    brand_mask = explicit_brand_facet_mask(facets)
    review_mask = explicit_review_facet_mask(facets)
    global_mask = metadata_mask | brand_mask | review_mask

    core_mismatch_rows = int(facets["is_core_graph_facet"].ne(metadata_mask).sum())
    brand_mismatch_rows = int(facets["is_brand_graph_facet"].ne(brand_mask).sum())
    global_mismatch_rows = int(facets["is_global_review_graph_facet"].ne(global_mask).sum())
    if core_mismatch_rows:
        raise RuntimeError(
            f"is_core_graph_facet disagrees with the explicit metadata mask: {core_mismatch_rows}"
        )
    if brand_mismatch_rows:
        raise RuntimeError(
            f"is_brand_graph_facet disagrees with the explicit brand mask: {brand_mismatch_rows}"
        )
    if global_mismatch_rows:
        raise RuntimeError(
            f"is_global_review_graph_facet disagrees with the explicit Global Review mask: {global_mismatch_rows}"
        )

    metadata_rows = facets.loc[metadata_mask].copy()
    brand_rows = facets.loc[brand_mask].copy()
    review_rows = facets.loc[review_mask].copy()
    global_rows = facets.loc[global_mask].copy()
    if metadata_rows.empty:
        raise RuntimeError("Metadata product-functional facet rows must be greater than zero.")
    if brand_rows.empty:
        raise RuntimeError("Brand facet rows must be greater than zero.")
    if review_rows.empty:
        raise RuntimeError("Historical review-derived facet rows must be greater than zero.")
    if global_rows.empty:
        raise RuntimeError("Global Review graph facet rows must be greater than zero.")

    source_text = metadata_rows["source_column"].fillna("").astype(str).str.lower()
    disallowed_source_rows = source_text.map(
        lambda value: any(term in value for term in FORBIDDEN_FACET_SOURCE_TERMS)
    )
    if disallowed_source_rows.any():
        raise RuntimeError(
            "Identifier, count, policy, or diagnostic sources entered the metadata graph mask."
        )
    if brand_rows["is_query_safe"].any():
        raise RuntimeError("Brand rows must remain query-unsafe.")
    if brand_rows["is_product_functional_facet"].any():
        raise RuntimeError("Brand rows must remain separate from product-functional facets.")
    if not brand_rows["is_retrieval_safe"].all():
        raise RuntimeError("Brand rows must be retrieval-safe.")
    if not brand_rows["is_profile_safe"].all():
        raise RuntimeError("Brand rows must be profile-safe.")
    if review_rows["is_product_functional_facet"].any():
        raise RuntimeError("Historical review-derived rows must not be product-functional facets.")
    if review_rows["is_query_safe"].any():
        raise RuntimeError("Historical review-derived rows must not be query-safe facets.")

    return pd.DataFrame([{
        "raw_facet_rows": int(len(facets)),
        "global_review_flagged_rows": int(len(global_rows)),
        "metadata_functional_rows": int(len(metadata_rows)),
        "brand_rows_used": int(len(brand_rows)),
        "review_derived_rows_used": int(len(review_rows)),
        "generic_anchor_rows_used": int(global_rows["is_generic_category_anchor"].sum()),
        "generic_utility_rows_used": int(global_rows["is_generic_utility_token"].sum()),
        "context_utility_rows_used": int(global_rows["is_context_dependent_utility_token"].sum()),
        "disallowed_source_rows_used": int(global_rows["is_disallowed_nonfacet_source"].sum()),
        "core_mask_mismatch_rows": core_mismatch_rows,
        "brand_mask_mismatch_rows": brand_mismatch_rows,
        "global_mask_mismatch_rows": global_mismatch_rows,
        "historical_facet_values_loaded": int(len(review_rows)),
        "brand_facet_values_loaded": int(len(brand_rows)),
        "brand_query_matching_enabled": False,
        "global_value_load_filter": "is_global_review_graph_facet == True",
        "item_facet_export_option": "B_all_rows_with_reliable_flags",
        "status": "PASS",
    }])

def canonicalize_graph_facets(raw_facets, item_to_position, facet_kind):
    required = [
        FACET_ITEM_ID_COLUMN,
        FACET_VALUE_COLUMN,
        "source_column",
        "facet_role",
        *GRAPH_FACET_FLAG_COLUMNS,
    ]
    require_columns(raw_facets, required, f"{facet_kind} item facets")
    facets = normalize_facet_flags(raw_facets[required])
    if facets.empty:
        raise RuntimeError(f"{facet_kind} item-facet value rows must be greater than zero.")

    masks = {
        "metadata": explicit_metadata_facet_mask,
        "brand": explicit_brand_facet_mask,
        "review": explicit_review_facet_mask,
    }
    if facet_kind not in masks:
        raise RuntimeError(f"Unsupported graph facet kind: {facet_kind}")
    expected_mask = masks[facet_kind](facets)
    if not expected_mask.all():
        raise RuntimeError(f"A loaded {facet_kind} facet row violates its explicit mask.")
    if not facets["is_global_review_graph_facet"].all():
        raise RuntimeError(f"A loaded {facet_kind} facet row is outside the Global Review graph mask.")

    facets["item_id"] = facets[FACET_ITEM_ID_COLUMN].astype(str)
    facets["item_position"] = facets["item_id"].map(item_to_position)
    if facets["item_position"].isna().any():
        raise RuntimeError(f"{facet_kind} facets contain item IDs missing from item docs.")
    facets["item_position"] = facets["item_position"].astype(np.int32)
    facets["facet_value_norm"] = facets[FACET_VALUE_COLUMN].map(canonical_text)
    facets = facets.loc[facets["facet_value_norm"].str.len().gt(0)].copy()
    facets = facets.drop_duplicates(["item_position", "facet_value_norm"]).reset_index(drop=True)
    if facets.empty:
        raise RuntimeError(f"No non-empty {facet_kind} facet values remain after normalization.")
    return facets


def validate_global_review_graph_edges(graph_edges):
    required = [
        "source_node_type",
        "target_node_type",
        "source_column",
        "facet_role",
        *GRAPH_FACET_FLAG_COLUMNS,
    ]
    require_columns(graph_edges, required, "Global Review graph edges")
    graph_edges = normalize_facet_flags(graph_edges[required])

    metadata_mask = explicit_metadata_facet_mask(graph_edges)
    brand_mask = explicit_brand_facet_mask(graph_edges)
    review_mask = explicit_review_facet_mask(graph_edges)
    global_mask = metadata_mask | brand_mask | review_mask

    if not graph_edges["is_core_graph_facet"].eq(metadata_mask).all():
        raise RuntimeError("Graph-edge is_core_graph_facet disagrees with the metadata mask.")
    if not graph_edges["is_brand_graph_facet"].eq(brand_mask).all():
        raise RuntimeError("Graph-edge is_brand_graph_facet disagrees with the brand mask.")
    if not graph_edges["is_global_review_graph_facet"].eq(global_mask).all():
        raise RuntimeError("Graph-edge is_global_review_graph_facet disagrees with the Global Review mask.")
    if not graph_edges["is_global_review_graph_facet"].all():
        raise RuntimeError("The production graph contains a row outside the Global Review graph mask.")
    if not graph_edges["source_node_type"].eq("item").all():
        raise RuntimeError("Production graph source nodes must be items.")
    if not graph_edges["target_node_type"].eq("entity").all():
        raise RuntimeError("Production graph target nodes must be entities.")
    if int(metadata_mask.sum()) <= 0:
        raise RuntimeError("Metadata graph edges must be greater than zero.")
    if int(brand_mask.sum()) <= 0:
        raise RuntimeError("Brand graph edges must be greater than zero.")
    if int(review_mask.sum()) <= 0:
        raise RuntimeError("Historical review graph edges must be greater than zero.")
    if int(global_mask.sum()) != len(graph_edges):
        raise RuntimeError("Global Review graph edge decomposition is incomplete.")

    for column in [
        "is_generic_category_anchor",
        "is_generic_utility_token",
        "is_context_dependent_utility_token",
        "is_disallowed_nonfacet_source",
    ]:
        if graph_edges[column].any():
            raise RuntimeError(f"Production graph contains prohibited rows: {column}")
    if not graph_edges.loc[metadata_mask, "is_product_functional_facet"].all():
        raise RuntimeError("Metadata graph rows must be product-functional.")
    if not graph_edges.loc[metadata_mask, "is_query_safe"].all():
        raise RuntimeError("Metadata graph rows must be query-safe.")
    if graph_edges.loc[brand_mask, "is_query_safe"].any():
        raise RuntimeError("Brand graph rows must remain query-unsafe.")
    if graph_edges.loc[brand_mask, "is_product_functional_facet"].any():
        raise RuntimeError("Brand graph rows must remain separate from product-functional facets.")
    if not graph_edges.loc[brand_mask, "is_retrieval_safe"].all():
        raise RuntimeError("Brand graph rows must be retrieval-safe.")
    if not graph_edges.loc[brand_mask, "is_profile_safe"].all():
        raise RuntimeError("Brand graph rows must be profile-safe.")
    if graph_edges.loc[review_mask, "is_product_functional_facet"].any():
        raise RuntimeError("Historical review graph rows must not be product-functional.")
    if graph_edges.loc[review_mask, "is_query_safe"].any():
        raise RuntimeError("Historical review graph rows must not be query-safe.")


def tokenize_sparse_document(value):
    return [token for token in canonical_tokens(value) if token not in LINGUISTIC_STOPWORDS]


def tokenize_sparse_query(value):
    stopwords = LINGUISTIC_STOPWORDS | GENERIC_ANCHOR_TOKENS
    return [token for token in canonical_tokens(value) if token not in stopwords]


def select_graph_query_text(active_query_text):
    tokens = [token for token in canonical_tokens(active_query_text) if token not in GRAPH_STOPWORDS]
    return " ".join(tokens)


def build_graph_index(facets, n_items):
    phrase_items = defaultdict(set)
    token_items = defaultdict(set)
    for row in facets[["item_position", "facet_value_norm"]].itertuples(index=False):
        item_position = int(row.item_position)
        tokens = canonical_tokens(row.facet_value_norm)
        if not tokens:
            continue
        if 2 <= len(tokens) <= GRAPH_MAX_PHRASE_TOKENS:
            phrase_items[" ".join(tokens)].add(item_position)
        for token in dict.fromkeys(tokens):
            if token in GRAPH_STOPWORDS:
                continue
            if len(token) < MIN_GRAPH_TOKEN_LENGTH and token not in GRAPH_SHORT_TOKENS:
                continue
            token_items[token].add(item_position)
    phrase_idf = {
        phrase: math.log((n_items + 1) / (len(items) + 1)) + 1.0
        for phrase, items in phrase_items.items()
    }
    token_idf = {
        token: math.log((n_items + 1) / (len(items) + 1)) + 1.0
        for token, items in token_items.items()
    }
    return {
        "phrase_items": phrase_items,
        "token_items": token_items,
        "phrase_idf": phrase_idf,
        "token_idf": token_idf,
    }


def graph_score_map(raw_query_text, graph_query_text, graph_index):
    raw_tokens = canonical_tokens(raw_query_text)
    graph_tokens = canonical_tokens(graph_query_text)
    matched_raw_positions = set()
    matched_phrase_tokens = set()
    scores = defaultdict(float)

    max_phrase_length = min(GRAPH_MAX_PHRASE_TOKENS, len(raw_tokens))
    for phrase_length in range(max_phrase_length, 1, -1):
        for start in range(len(raw_tokens) - phrase_length + 1):
            positions = set(range(start, start + phrase_length))
            if positions.intersection(matched_raw_positions):
                continue
            phrase_tokens = raw_tokens[start:start + phrase_length]
            if all(token in GRAPH_STOPWORDS for token in phrase_tokens):
                continue
            phrase = " ".join(phrase_tokens)
            postings = graph_index["phrase_items"].get(phrase)
            if not postings:
                continue
            contribution = GRAPH_PHRASE_BONUS * graph_index["phrase_idf"][phrase]
            for item_position in postings:
                scores[item_position] += contribution
            matched_raw_positions.update(positions)
            matched_phrase_tokens.update(phrase_tokens)

    for token in dict.fromkeys(graph_tokens):
        if token in matched_phrase_tokens or token in GRAPH_STOPWORDS:
            continue
        if len(token) < MIN_GRAPH_TOKEN_LENGTH and token not in GRAPH_SHORT_TOKENS:
            continue
        postings = graph_index["token_items"].get(token)
        if not postings:
            continue
        contribution = graph_index["token_idf"][token]
        for item_position in postings:
            scores[item_position] += contribution
    return scores


def top_items_from_scores(score_map, top_k):
    if not score_map:
        return []
    return [
        int(item_position)
        for item_position, _ in heapq.nsmallest(
            min(top_k, len(score_map)),
            score_map.items(),
            key=lambda pair: (-pair[1], pair[0]),
        )
    ]


def bm25_top_exact(bm25, query_tokens, top_k):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float64)
    if len(scores) == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float64)
    if not np.isfinite(scores).all():
        raise RuntimeError("BM25 scores contain non-finite values.")
    k_eff = min(int(top_k), len(scores))
    stable_position = np.arange(len(scores), dtype=np.int64)
    order = np.lexsort((stable_position, -scores))
    selected = order[:k_eff].astype(np.int64)
    return selected, scores[selected]


class _SyntheticBM25:
    def __init__(self, scores):
        self._scores = np.asarray(scores, dtype=np.float64)

    def get_scores(self, query_tokens):
        return self._scores.copy()


def validate_bm25_top_exact_helper():
    cases = [
        ("zero_positive", [0.0, 0.0, 0.0, 0.0], 3),
        ("fewer_than_k_positive", [3.0, 1.0, 0.0, 0.0], 3),
        ("exactly_k_positive", [3.0, 2.0, 1.0, 0.0], 3),
        ("more_than_k_positive", [4.0, 3.0, 2.0, 1.0], 3),
        ("tied_zero_scores", [0.0, 0.0, 0.0, 0.0, 0.0], 4),
        ("negative_scores", [1.0, 0.0, -0.5, -0.5, -1.0], 5),
        ("catalog_smaller_than_k", [2.0, 0.0], 5),
    ]
    for name, scores, top_k in cases:
        bm25 = _SyntheticBM25(scores)
        first_idx, first_scores = bm25_top_exact(bm25, ["query"], top_k)
        second_idx, second_scores = bm25_top_exact(bm25, ["query"], top_k)
        expected_k = min(top_k, len(scores))
        if len(first_idx) != expected_k:
            raise RuntimeError(f"BM25 exact-K helper failed candidate count test: {name}")
        if not np.array_equal(first_idx, second_idx) or not np.array_equal(first_scores, second_scores):
            raise RuntimeError(f"BM25 exact-K helper is not deterministic: {name}")
        expected_order = np.lexsort((np.arange(len(scores), dtype=np.int64), -np.asarray(scores, dtype=np.float64)))[:expected_k]
        if not np.array_equal(first_idx, expected_order):
            raise RuntimeError(f"BM25 exact-K helper ordering mismatch: {name}")


validate_bm25_top_exact_helper()


def rank_map(items):
    return {int(item): rank for rank, item in enumerate(items, start=1)}


def reciprocal_rank_fusion(rank_maps, weights, top_k):
    scores = defaultdict(float)
    for source_name, source_rank_map in rank_maps.items():
        weight = float(weights[source_name])
        for item_position, rank in source_rank_map.items():
            scores[item_position] += weight / (RRF_K + rank)
    ranked_pairs = heapq.nsmallest(
        min(top_k, len(scores)),
        scores.items(),
        key=lambda pair: (-pair[1], pair[0]),
    )
    return [int(item) for item, _ in ranked_pairs], dict(scores)


def target_rank(ranked_items, target_position):
    for rank, item_position in enumerate(ranked_items, start=1):
        if item_position == target_position:
            return rank
    return None


def metric_record(rank):
    record = {}
    for k in EVAL_KS:
        hit = float(rank is not None and rank <= k)
        record[f"HitRate@{k}"] = hit
        record[f"NDCG@{k}"] = 1.0 / math.log2(rank + 1) if hit else 0.0
        record[f"MRR@{k}"] = 1.0 / rank if hit else 0.0
    return record


def aggregate_metrics(frame, group_columns):
    metric_columns = [
        column for column in frame.columns
        if column.startswith("HitRate@") or column.startswith("NDCG@") or column.startswith("MRR@")
    ]
    grouped = frame.groupby(group_columns, observed=True, dropna=False)
    result = grouped[metric_columns].mean().reset_index()
    counts = grouped.size().rename("n_queries").reset_index()
    return counts.merge(result, on=group_columns, how="left")


def pool_depth_summary(frame, group_columns):
    rows = []
    for group_values, group_frame in frame.groupby(group_columns, observed=True, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        group_dict = dict(zip(group_columns, group_values))
        for k in EVAL_KS:
            rows.append({
                **group_dict,
                "candidate_pool_depth": k,
                "n_queries": int(len(group_frame)),
                "HitRate": float(group_frame[f"HitRate@{k}"].mean()),
                "NDCG": float(group_frame[f"NDCG@{k}"].mean()),
                "MRR": float(group_frame[f"MRR@{k}"].mean()),
            })
    return pd.DataFrame(rows)


CANDIDATE_COLUMNS = [
    "case_id",
    "user_id",
    "regime",
    "query_method",
    "method_key",
    "retrieval_method",
    "evidence_scope",
    "target_parent_asin",
    "candidate_parent_asin",
    "candidate_brand_facet_text",
    "candidate_rank",
    "candidate_score",
    "fused_score",
    "score_semantics",
    "dense_rank",
    "bm25_rank",
    "metadata_graph_rank",
    "review_reputation_graph_rank",
    "is_target",
]


def candidate_frame(rows):
    frame = pd.DataFrame(rows, columns=CANDIDATE_COLUMNS)
    string_columns = [
        "case_id", "user_id", "regime", "query_method", "method_key",
        "retrieval_method", "evidence_scope", "target_parent_asin",
        "candidate_parent_asin", "candidate_brand_facet_text", "score_semantics",
    ]
    for column in string_columns:
        frame[column] = frame[column].astype("string")
    for column in [
        "candidate_rank",
        "dense_rank",
        "bm25_rank",
        "metadata_graph_rank",
        "review_reputation_graph_rank",
    ]:
        frame[column] = frame[column].astype("Int32")
    for column in ["candidate_score", "fused_score"]:
        frame[column] = frame[column].astype(np.float32)
    frame["is_target"] = frame["is_target"].astype(bool)
    return frame


In [ ]:
# =========================================================
# Inputs and Validation
# =========================================================
load_start = time.perf_counter()

required_paths = [
    QUERY_CACHE_PATH,
    QUERY_CONTRACT_PATH,
    ITEM_DOCS_PATH,
    ITEM_FACETS_PATH,
    GRAPH_EDGES_PATH,
    RETRIEVAL_ARTIFACT_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

query_contract = load_json(QUERY_CONTRACT_PATH)
retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
validate_query_contract(query_contract)
validate_retrieval_manifest(retrieval_manifest)

query_schema_columns = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
query_alias_columns = sorted(KNOWN_QUERY_ALIAS_COLUMNS.intersection(query_schema_columns))
expected_query_alias_columns = [] if COMPATIBILITY_QUERY_ALIAS is None else [COMPATIBILITY_QUERY_ALIAS]
if query_alias_columns != expected_query_alias_columns:
    raise RuntimeError(
        "Active query aliases are ambiguous: "
        f"expected {expected_query_alias_columns}, found {query_alias_columns}."
    )
required_query_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "target_selection_mode",
    "target_timestamp_ms",
    ACTIVE_QUERY_COLUMN,
    "query_evidence_source",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "item_metadata_evidence_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "rating_evidence_used",
    "sentiment_evidence_used",
    "insufficient_review_evidence",
    "query_generation_status",
    "query_specific_facet_family_count",
    "query_generic_anchor_terms",
    "query_generic_utility_terms",
    "query_generic_anchor_rate",
    "query_specific_facet_cue_rate",
    "query_clean_is_active",
    *QUERY_PASSTHROUGH_COLUMNS,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    required_query_columns.append(COMPATIBILITY_QUERY_ALIAS)
missing_query_columns = [column for column in required_query_columns if column not in query_schema_columns]
if missing_query_columns:
    raise RuntimeError(f"Query cache is missing required columns: {missing_query_columns}")
query_load_columns = required_query_columns + [
    column for column in OPTIONAL_QUERY_AUDIT_COLUMNS
    if column in query_schema_columns and column not in required_query_columns
]
query_load_columns = list(dict.fromkeys(query_load_columns))
queries_full = pd.read_parquet(QUERY_CACHE_PATH, columns=query_load_columns).copy()
sampled_rank_df = pd.read_parquet(
    SAMPLED_USERS_PATH,
    columns=["case_id", "target_rank_desc"],
).copy()

eligible_rank_df = pd.read_parquet(
    FINAL_SAMPLING_POOL_PATH,
    columns=["case_id", "target_rank_desc"],
).copy()

target_rank_lookup_df = (
    pd.concat([sampled_rank_df, eligible_rank_df], ignore_index=True)
    .dropna(subset=["case_id"])
    .copy()
)
target_rank_lookup_df["case_id"] = target_rank_lookup_df["case_id"].astype(str).str.strip()
target_rank_lookup_df["target_rank_desc"] = pd.to_numeric(
    target_rank_lookup_df["target_rank_desc"],
    errors="coerce",
)
target_rank_lookup_df = (
    target_rank_lookup_df
    .dropna(subset=["target_rank_desc"])
    .drop_duplicates("case_id", keep="first")
)

queries_full["case_id"] = queries_full["case_id"].astype(str).str.strip()
queries_full = queries_full.merge(
    target_rank_lookup_df,
    on="case_id",
    how="left",
    validate="many_to_one",
)

if queries_full["target_rank_desc"].isna().any():
    missing_rank_cases = queries_full.loc[
        queries_full["target_rank_desc"].isna(),
        ["case_id", "replacement_case_used", "replacement_source_case_id"],
    ].head(20)
    display(missing_rank_cases)
    raise RuntimeError("target_rank_desc must be non-null after sampled + eligible pool merge.")

item_docs = pd.read_parquet(
    ITEM_DOCS_PATH,
    columns=[ITEM_ID_COLUMN, DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN],
).copy()
facet_flag_columns = [
    FACET_ITEM_ID_COLUMN,
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
facet_value_columns = [
    FACET_ITEM_ID_COLUMN,
    FACET_VALUE_COLUMN,
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
facet_schema_columns = pq.ParquetFile(ITEM_FACETS_PATH).schema.names
missing_facet_columns = [
    column for column in facet_value_columns
    if column not in facet_schema_columns
]
if missing_facet_columns:
    raise RuntimeError(f"Item facets are missing required columns: {missing_facet_columns}")
item_facet_flags = pd.read_parquet(
    ITEM_FACETS_PATH,
    columns=facet_flag_columns,
).copy()
facet_filter_audit_df = audit_global_review_facet_flags(item_facet_flags)
metadata_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[("is_core_graph_facet", "=", True)],
).to_pandas()
brand_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[("is_brand_graph_facet", "=", True)],
).to_pandas()
review_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[
        ("is_review_derived", "=", True),
        ("is_global_review_graph_facet", "=", True),
    ],
).to_pandas()
graph_edge_columns = [
    "source_node_type",
    "target_node_type",
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
graph_edges = pd.read_parquet(GRAPH_EDGES_PATH, columns=graph_edge_columns).copy()
validate_global_review_graph_edges(graph_edges)

EXPECTED_ROWS = int(len(queries_full))
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    if queries_full[column].isna().any() or queries_full[column].map(normalize_space).eq("").any():
        raise RuntimeError(f"{column} must be non-null and non-empty.")
if queries_full["case_id"].astype(str).duplicated().any():
    raise RuntimeError("case_id must be unique.")
if queries_full["user_id"].astype(str).duplicated().any():
    raise RuntimeError("user_id must be unique.")
if queries_full["target_timestamp_ms"].isna().any():
    raise RuntimeError("target_timestamp_ms must be non-null.")
if not queries_full["target_selection_mode"].fillna("").astype(str).eq(EXPECTED_TARGET_SELECTION_MODE).all():
    observed = sorted(queries_full["target_selection_mode"].astype(str).unique().tolist())
    raise RuntimeError(f"Unexpected target_selection_mode values: {observed}")
if MAX_TARGET_RANK_ALLOWED is not None:
    if queries_full["target_rank_desc"].isna().any():
        raise RuntimeError("target_rank_desc must be non-null.")
    if queries_full["target_rank_desc"].astype(int).gt(MAX_TARGET_RANK_ALLOWED).any():
        raise RuntimeError(f"target_rank_desc must not exceed {MAX_TARGET_RANK_ALLOWED}.")

observed_regime_counts = (
    queries_full["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
contract_target_per_regime = int(query_contract.get("target_per_regime", -1))
expected_contract_counts = {regime: int(contract_target_per_regime) for regime in REGIME_ORDER}
if observed_regime_counts != expected_contract_counts:
    raise RuntimeError(
        f"Regime count mismatch against Notebook 06 contract: actual {observed_regime_counts}, "
        f"expected {expected_contract_counts}"
    )
EXPECTED_REGIME_COUNTS = dict(observed_regime_counts)

active_query = queries_full[ACTIVE_QUERY_COLUMN].map(normalize_space)
if active_query.eq("").any():
    raise RuntimeError("The active query must be non-empty for every row.")
if COMPATIBILITY_QUERY_ALIAS is not None:
    compatibility_query = queries_full[COMPATIBILITY_QUERY_ALIAS].map(normalize_space)
    if not compatibility_query.eq(active_query).all():
        raise RuntimeError(f"{COMPATIBILITY_QUERY_ALIAS} must be an exact alias of query.")
if boolean_series(queries_full["query_clean_is_active"]).any():
    raise RuntimeError("query_clean must remain inactive for the production retrieval run.")
if not queries_full["query_evidence_source"].fillna("").astype(str).eq("target_review_safe_signals_only").all():
    raise RuntimeError("Every query must use target_review_safe_signals_only evidence.")
validate_false_columns(
    queries_full,
    [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "insufficient_review_evidence",
    ],
    "query cache",
)
if queries_full["query_specific_facet_family_count"].fillna(0).astype(int).le(0).any():
    raise RuntimeError("Every final query must retain at least one specific review-derived cue family.")
if queries_full["query_generation_status"].isna().any() or queries_full["query_generation_status"].map(normalize_space).eq("").any():
    raise RuntimeError("query_generation_status must be non-empty for every row.")

forbidden_loaded_query_columns = {
    "raw_review_text",
    "target_review_text",
    "review_title",
    "review_body",
    "brand",
    "title",
    "manufacturer",
    "seller",
    "query_safe_facet_text",
    "common_functional_facet_text",
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "prior_review_text",
    "prior_history_n",
}
loaded_forbidden = sorted(forbidden_loaded_query_columns.intersection(queries_full.columns))
if loaded_forbidden:
    raise RuntimeError(f"Prohibited query evidence columns were loaded: {loaded_forbidden}")

if item_docs[ITEM_ID_COLUMN].isna().any() or item_docs[ITEM_ID_COLUMN].map(normalize_space).eq("").any():
    raise RuntimeError("Item docs must contain non-null, non-empty parent_asin values.")
item_docs[ITEM_ID_COLUMN] = item_docs[ITEM_ID_COLUMN].astype(str)
if item_docs[ITEM_ID_COLUMN].duplicated().any():
    raise RuntimeError("Item docs must contain one row per parent_asin.")
if len(item_docs) == 0:
    raise RuntimeError("Global Review catalog must contain at least one item.")
for column in [DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN]:
    if item_docs[column].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every item.")
item_docs[BRAND_TEXT_COLUMN] = item_docs[BRAND_TEXT_COLUMN].fillna("").astype(str).map(normalize_space)
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("The Global Review catalog must retain non-empty brand facets.")

item_docs = item_docs.sort_values(ITEM_ID_COLUMN, kind="stable").reset_index(drop=True)
catalog_size = int(len(item_docs))
EXPECTED_CANDIDATE_K = min(MAX_RETRIEVAL_K, catalog_size)
if EXPECTED_CANDIDATE_K <= 0:
    raise RuntimeError("Exact-K candidate budget must be positive.")
item_ids = item_docs[ITEM_ID_COLUMN].to_numpy(dtype=object)
item_brand_texts = item_docs[BRAND_TEXT_COLUMN].to_numpy(dtype=object)
item_id_set = set(item_ids)
item_to_position = {item_id: position for position, item_id in enumerate(item_ids)}
target_positions = queries_full["target_parent_asin"].astype(str).map(item_to_position)
if target_positions.isna().any():
    raise RuntimeError("At least one target item is missing from the Global Review catalog.")
queries_full["target_item_position"] = target_positions.astype(np.int32)

core_facets = canonicalize_graph_facets(metadata_item_facets_raw, item_to_position, "metadata")
brand_facets = canonicalize_graph_facets(brand_item_facets_raw, item_to_position, "brand")
review_facets = canonicalize_graph_facets(review_item_facets_raw, item_to_position, "review")
facet_filter_audit_df["metadata_functional_rows"] = int(len(core_facets))
facet_filter_audit_df["metadata_facet_items"] = int(core_facets["item_position"].nunique())
facet_filter_audit_df["brand_rows_used"] = int(len(brand_facets))
facet_filter_audit_df["brand_facet_items"] = int(brand_facets["item_position"].nunique())
facet_filter_audit_df["review_derived_rows"] = int(len(review_facets))
facet_filter_audit_df["review_facet_items"] = int(review_facets["item_position"].nunique())
facet_filter_audit_df["metadata_graph_edges"] = int(
    graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum()
)
facet_filter_audit_df["brand_graph_edges"] = int(
    graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum()
)
facet_filter_audit_df["review_reputation_graph_edges"] = int(
    graph_edges["is_review_derived"].fillna(False).astype(bool).sum()
)

queries_full["active_query_text"] = active_query
queries_full["query_text_retrieval"] = queries_full["active_query_text"]
queries_full["query_text_graph"] = queries_full["active_query_text"].map(select_graph_query_text)
if queries_full["query_text_retrieval"].map(canonical_tokens).map(len).eq(0).any():
    raise RuntimeError("Dense/BM25 query text must contain at least one token.")
if queries_full["query_text_graph"].map(canonical_tokens).map(len).eq(0).any():
    raise RuntimeError("A final strict query produced no graph-safe tokens.")
if not queries_full["query_text_retrieval"].eq(queries_full["active_query_text"]).all():
    raise RuntimeError("Dense and BM25 must use the exact active Notebook 06 query.")
queries_full["query_method"] = QUERY_METHOD_LABEL

if RUN_SMOKE_TEST:
    if SMOKE_TEST_N > EXPECTED_ROWS:
        raise RuntimeError("SMOKE_TEST_N cannot exceed EXPECTED_ROWS.")
    evaluation_queries_df = (
        queries_full.sample(n=SMOKE_TEST_N, random_state=RANDOM_SEED)
        .sort_values(["regime", "case_id"], kind="stable")
        .reset_index(drop=True)
    )
else:
    evaluation_queries_df = queries_full.copy().reset_index(drop=True)

load_validation_runtime_sec = time.perf_counter() - load_start
print("Rows: queries", len(evaluation_queries_df))
print("Rows: items", len(item_docs))
print("Rows: metadata facets", len(core_facets))
print("Rows: brand facets", len(brand_facets))
print("Rows: historical review facets", len(review_facets))
print("Rows: Global Review graph edges", len(graph_edges))
print("Validation: inputs passed")


In [ ]:
# =========================================================
# Retrieval Indexes
# =========================================================
offline_index_start = time.perf_counter()

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
item_embeddings = model.encode(
    item_docs[DENSE_TEXT_COLUMN].astype(str).tolist(),
    batch_size=ITEM_EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
dense_index = faiss.IndexFlatIP(item_embeddings.shape[1])
dense_index.add(np.ascontiguousarray(item_embeddings))

sparse_corpus_tokens = [
    tokenize_sparse_document(text)
    for text in item_docs[SPARSE_TEXT_COLUMN].astype(str)
]
if any(len(tokens) == 0 for tokens in sparse_corpus_tokens):
    raise RuntimeError("Every production sparse item document must contain an indexed token.")
bm25 = BM25Okapi(sparse_corpus_tokens)

metadata_graph_index = build_graph_index(core_facets, len(item_docs))
review_reputation_graph_index = build_graph_index(review_facets, len(item_docs))
if not metadata_graph_index["token_items"] and not metadata_graph_index["phrase_items"]:
    raise RuntimeError("The metadata functional-facet graph index is empty.")
if not review_reputation_graph_index["token_items"] and not review_reputation_graph_index["phrase_items"]:
    raise RuntimeError("The historical review-derived graph index is empty.")

offline_index_runtime_sec = time.perf_counter() - offline_index_start

query_embedding_start = time.perf_counter()
query_embeddings = model.encode(
    evaluation_queries_df["query_text_retrieval"].astype(str).tolist(),
    batch_size=QUERY_EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
query_embedding_runtime_sec = time.perf_counter() - query_embedding_start

dense_search_start = time.perf_counter()
dense_scores, dense_positions = dense_index.search(
    np.ascontiguousarray(query_embeddings),
    EXPECTED_CANDIDATE_K,
)
dense_search_runtime_sec = time.perf_counter() - dense_search_start
if dense_positions.shape != (len(evaluation_queries_df), EXPECTED_CANDIDATE_K):
    raise RuntimeError("Unexpected Dense BERT Core output shape.")
if (dense_positions < 0).any():
    raise RuntimeError("Dense BERT Core returned invalid item positions.")

print("Rows: dense index", dense_index.ntotal)
print("Rows: metadata graph phrases", len(metadata_graph_index["phrase_items"]))
print("Rows: brand graph facets preserved", len(brand_facets))
print("Brand query matching enabled:", BRAND_QUERY_MATCHING_ENABLED)
print("Rows: review graph phrases", len(review_reputation_graph_index["phrase_items"]))
print("Rows: metadata graph tokens", len(metadata_graph_index["token_items"]))
print("Rows: review graph tokens", len(review_reputation_graph_index["token_items"]))
print("Validation: indexes passed")

In [ ]:
# =========================================================
# Retrieval Evaluation and Candidate Export
# =========================================================
candidate_temp_paths = {
    method_key: path.with_suffix(".tmp.parquet")
    for method_key, path in METHOD_CANDIDATE_PATHS.items()
}
for temp_path in candidate_temp_paths.values():
    if temp_path.exists():
        temp_path.unlink()

empty_candidate_table = pa.Table.from_pandas(candidate_frame([]), preserve_index=False)
candidate_writers = {
    method_key: pq.ParquetWriter(candidate_temp_paths[method_key], empty_candidate_table.schema)
    for method_key in METHOD_KEYS
}
candidate_batches = {method_key: [] for method_key in METHOD_KEYS}
candidate_row_counts = {method_key: 0 for method_key in METHOD_KEYS}
candidate_target_hit_counts = {method_key: 0 for method_key in METHOD_KEYS}
candidate_write_runtime_sec = {method_key: 0.0 for method_key in METHOD_KEYS}

per_query_rows = []
diagnostic_rows = []
component_runtime = defaultdict(float)
component_runtime["query_embedding"] = query_embedding_runtime_sec
component_runtime["dense_global_review_faiss"] = dense_search_runtime_sec

evaluation_start = time.perf_counter()
for query_index, query_row in enumerate(
    tqdm(evaluation_queries_df.itertuples(index=False), total=len(evaluation_queries_df))
):
    dense_items = dense_positions[query_index].astype(np.int32).tolist()
    dense_rank_map = rank_map(dense_items)
    dense_score_map = {
        int(item_position): float(score)
        for item_position, score in zip(dense_items, dense_scores[query_index].tolist())
    }

    bm25_start = time.perf_counter()
    bm25_query_tokens = tokenize_sparse_query(query_row.query_text_retrieval)
    bm25_selected, bm25_selected_scores = bm25_top_exact(bm25, bm25_query_tokens, EXPECTED_CANDIDATE_K)
    bm25_items = bm25_selected.astype(np.int32).tolist()
    bm25_score_map = {
        int(item_position): float(score)
        for item_position, score in zip(bm25_items, bm25_selected_scores.tolist())
    }
    bm25_positive_score_candidate_count = int((bm25_selected_scores > 0).sum())
    bm25_zero_score_selected_count = int((bm25_selected_scores == 0).sum())
    bm25_negative_score_selected_count = int((bm25_selected_scores < 0).sum())
    component_runtime["bm25_global_review_retrieval"] += time.perf_counter() - bm25_start
    bm25_rank_map = rank_map(bm25_items)

    hybrid_start = time.perf_counter()
    hybrid_items, hybrid_scores = reciprocal_rank_fusion(
        {"dense": dense_rank_map, "bm25": bm25_rank_map},
        HYBRID_WEIGHTS,
        EXPECTED_CANDIDATE_K,
    )
    component_runtime["dense_bm25_global_review_rrf"] += time.perf_counter() - hybrid_start

    graph_start = time.perf_counter()
    metadata_graph_scores = graph_score_map(
        query_row.active_query_text,
        query_row.query_text_graph,
        metadata_graph_index,
    )
    metadata_graph_items = top_items_from_scores(metadata_graph_scores, EXPECTED_CANDIDATE_K)
    component_runtime["metadata_functional_graph_retrieval"] += time.perf_counter() - graph_start
    metadata_graph_rank_map = rank_map(metadata_graph_items)

    review_graph_start = time.perf_counter()
    review_reputation_graph_scores = graph_score_map(
        query_row.active_query_text,
        query_row.query_text_graph,
        review_reputation_graph_index,
    )
    review_reputation_graph_items = top_items_from_scores(
        review_reputation_graph_scores,
        MAX_RETRIEVAL_K,
    )
    component_runtime["review_reputation_graph_retrieval"] += (
        time.perf_counter() - review_graph_start
    )
    review_reputation_graph_rank_map = rank_map(review_reputation_graph_items)

    graph_hybrid_start = time.perf_counter()
    graph_hybrid_items, graph_hybrid_scores = reciprocal_rank_fusion(
        {
            "dense": dense_rank_map,
            "bm25": bm25_rank_map,
            "metadata_graph": metadata_graph_rank_map,
            "review_reputation_graph": review_reputation_graph_rank_map,
        },
        GRAPH_HYBRID_WEIGHTS,
        EXPECTED_CANDIDATE_K,
    )
    component_runtime["graph_hybrid_global_review_rrf"] += time.perf_counter() - graph_hybrid_start

    method_rankings = {
        "dense_bert": dense_items,
        "bm25": bm25_items,
        "hybrid_dense_bm25": hybrid_items,
        "graph_hybrid": graph_hybrid_items,
    }
    for method_key, ranked_items in method_rankings.items():
        if len(ranked_items) != len(set(ranked_items)):
            raise RuntimeError(f"Duplicate candidates for {method_key}, case_id={query_row.case_id}.")
        if len(ranked_items) != EXPECTED_CANDIDATE_K:
            raise RuntimeError(
                f"{method_key} returned {len(ranked_items)} candidates for case_id={query_row.case_id}; "
                f"expected {EXPECTED_CANDIDATE_K}."
            )
        if any(item_position < 0 or item_position >= catalog_size for item_position in ranked_items):
            raise RuntimeError(f"{method_key} returned an item outside the sorted catalog.")

    target_position = int(query_row.target_item_position)
    for method_key, ranked_items in method_rankings.items():
        rank = target_rank(ranked_items, target_position)
        per_query_rows.append({
            "case_id": str(query_row.case_id),
            "user_id": str(query_row.user_id),
            "regime": str(query_row.regime),
            "query_method": QUERY_METHOD_LABEL,
            "method_key": method_key,
            "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
            "evidence_scope": EVIDENCE_SCOPE,
            "target_rank": rank,
            **metric_record(rank),
        })
        candidate_target_hit_counts[method_key] += int(rank is not None)

    diagnostic_row = {
        "case_id": str(query_row.case_id),
        "user_id": str(query_row.user_id),
        "regime": str(query_row.regime),
        "query_method": QUERY_METHOD_LABEL,
        "active_query_column": ACTIVE_QUERY_COLUMN,
        "active_query_text": str(query_row.active_query_text),
        "query_text_retrieval": str(query_row.query_text_retrieval),
        "query_text_graph": str(query_row.query_text_graph),
        "target_parent_asin": str(query_row.target_parent_asin),
        "bm25_query_token_count": len(bm25_query_tokens),
        "bm25_candidate_count": len(bm25_items),
        "bm25_positive_candidate_count": bm25_positive_score_candidate_count,
        "bm25_positive_score_candidate_count": bm25_positive_score_candidate_count,
        "bm25_nonpositive_fill_count": int(len(bm25_items) - bm25_positive_score_candidate_count),
        "bm25_minimum_selected_score": float(bm25_selected_scores.min()),
        "bm25_zero_score_selected_count": bm25_zero_score_selected_count,
        "bm25_negative_score_selected_count": bm25_negative_score_selected_count,
        "metadata_graph_candidate_count": len(metadata_graph_items),
        "review_reputation_graph_candidate_count": len(review_reputation_graph_items),
        "graph_hybrid_candidate_union_count": len(
            set(dense_items)
            | set(bm25_items)
            | set(metadata_graph_items)
            | set(review_reputation_graph_items)
        ),
        "dense_component_present": int(len(dense_items) > 0),
        "bm25_component_present": int(len(bm25_items) > 0),
        "metadata_graph_component_present": int(len(metadata_graph_items) > 0),
        "review_reputation_graph_component_present": int(
            len(review_reputation_graph_items) > 0
        ),
        "brand_catalog_channel_present": 1,
        "brand_query_matching_enabled": int(BRAND_QUERY_MATCHING_ENABLED),
    }
    for column in QUERY_STRUCTURE_COLUMNS + QUERY_PASSTHROUGH_COLUMNS:
        diagnostic_row[column] = getattr(query_row, column)
    for column in OPTIONAL_QUERY_AUDIT_COLUMNS:
        if hasattr(query_row, column):
            diagnostic_row[column] = getattr(query_row, column)
    diagnostic_rows.append(diagnostic_row)

    candidate_specs = {
        "dense_bert": {
            "ranked_items": dense_items,
            "scores": dense_score_map,
            "score_semantics": "cosine_similarity",
            "rank_sources": {"dense"},
            "fused": False,
        },
        "bm25": {
            "ranked_items": bm25_items,
            "scores": bm25_score_map,
            "score_semantics": "bm25_score_exact_k",
            "rank_sources": {"bm25"},
            "fused": False,
        },
        "hybrid_dense_bm25": {
            "ranked_items": hybrid_items,
            "scores": hybrid_scores,
            "score_semantics": "rrf_dense_global_review_bm25_global_review",
            "rank_sources": {"dense", "bm25"},
            "fused": True,
        },
        "graph_hybrid": {
            "ranked_items": graph_hybrid_items,
            "scores": graph_hybrid_scores,
            "score_semantics": "rrf_dense_bm25_metadata_graph_review_reputation_graph",
            "rank_sources": {"dense", "bm25", "metadata_graph", "review_reputation_graph"},
            "fused": True,
        },
    }
    for method_key, spec in candidate_specs.items():
        for candidate_rank, item_position in enumerate(spec["ranked_items"], start=1):
            score = float(spec["scores"][item_position])
            candidate_batches[method_key].append({
                "case_id": str(query_row.case_id),
                "user_id": str(query_row.user_id),
                "regime": str(query_row.regime),
                "query_method": QUERY_METHOD_LABEL,
                "method_key": method_key,
                "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
                "evidence_scope": EVIDENCE_SCOPE,
                "target_parent_asin": str(query_row.target_parent_asin),
                "candidate_parent_asin": str(item_ids[item_position]),
                "candidate_brand_facet_text": str(item_brand_texts[item_position]),
                "candidate_rank": candidate_rank,
                "candidate_score": score,
                "fused_score": score if spec["fused"] else np.nan,
                "score_semantics": spec["score_semantics"],
                "dense_rank": (
                    dense_rank_map.get(item_position)
                    if "dense" in spec["rank_sources"]
                    else None
                ),
                "bm25_rank": (
                    bm25_rank_map.get(item_position)
                    if "bm25" in spec["rank_sources"]
                    else None
                ),
                "metadata_graph_rank": (
                    metadata_graph_rank_map.get(item_position)
                    if "metadata_graph" in spec["rank_sources"]
                    else None
                ),
                "review_reputation_graph_rank": (
                    review_reputation_graph_rank_map.get(item_position)
                    if "review_reputation_graph" in spec["rank_sources"]
                    else None
                ),
                "is_target": item_position == target_position,
            })

    should_flush = (
        (query_index + 1) % CANDIDATE_EXPORT_QUERY_BATCH_SIZE == 0
        or query_index == len(evaluation_queries_df) - 1
    )
    if should_flush:
        for method_key in METHOD_KEYS:
            if not candidate_batches[method_key]:
                continue
            write_start = time.perf_counter()
            batch_frame = candidate_frame(candidate_batches[method_key])
            candidate_writers[method_key].write_table(
                pa.Table.from_pandas(batch_frame, preserve_index=False)
            )
            candidate_row_counts[method_key] += len(batch_frame)
            candidate_batches[method_key] = []
            candidate_write_runtime_sec[method_key] += time.perf_counter() - write_start

for method_key in METHOD_KEYS:
    candidate_writers[method_key].close()
    candidate_temp_paths[method_key].replace(METHOD_CANDIDATE_PATHS[method_key])

evaluation_loop_runtime_sec = time.perf_counter() - evaluation_start
per_query_metrics_df = pd.DataFrame(per_query_rows)
query_diagnostics_df = pd.DataFrame(diagnostic_rows)

print("Rows: per-query metrics", len(per_query_metrics_df))
print("Rows: candidates", candidate_row_counts)
print("Validation: retrieval evaluation completed")

In [ ]:
# =========================================================
# Summaries and Outputs
# =========================================================
method_order = [METHOD_DISPLAY_NAMES[key] for key in METHOD_KEYS]
per_query_metrics_df["retrieval_method"] = pd.Categorical(
    per_query_metrics_df["retrieval_method"],
    categories=method_order,
    ordered=True,
)

results_overall_df = aggregate_metrics(
    per_query_metrics_df,
    ["query_method", "method_key", "retrieval_method", "evidence_scope"],
).sort_values("retrieval_method", kind="stable")
results_by_regime_df = aggregate_metrics(
    per_query_metrics_df,
    ["query_method", "method_key", "retrieval_method", "evidence_scope", "regime"],
).sort_values(["retrieval_method", "regime"], kind="stable")
results_by_pool_depth_df = pool_depth_summary(
    per_query_metrics_df,
    ["query_method", "method_key", "retrieval_method", "evidence_scope"],
).sort_values(["retrieval_method", "candidate_pool_depth"], kind="stable")
results_by_regime_pool_depth_df = pool_depth_summary(
    per_query_metrics_df,
    ["query_method", "method_key", "retrieval_method", "evidence_scope", "regime"],
).sort_values(["retrieval_method", "regime", "candidate_pool_depth"], kind="stable")

query_diagnostics_df["generic_anchor_token_share_band"] = pd.cut(
    query_diagnostics_df["query_generic_anchor_rate"].astype(float),
    bins=[-0.001, 0.0, 0.25, 0.50, 1.0],
    labels=["0", "(0,0.25]", "(0.25,0.50]", "(0.50,1.00]"],
    include_lowest=True,
)
query_diagnostics_df["specific_family_band"] = pd.cut(
    query_diagnostics_df["query_specific_facet_family_count"].astype(float),
    bins=[-0.001, 0.0, 1.0, 2.0, np.inf],
    labels=["0", "1", "2", "3+"],
    include_lowest=True,
)
metrics_with_structure = per_query_metrics_df.merge(
    query_diagnostics_df[["case_id", "generic_anchor_token_share_band", "specific_family_band"]],
    on="case_id",
    how="left",
    validate="many_to_one",
)
results_by_query_structure_df = aggregate_metrics(
    metrics_with_structure,
    [
        "query_method",
        "method_key",
        "retrieval_method",
        "generic_anchor_token_share_band",
        "specific_family_band",
    ],
).sort_values(
    ["retrieval_method", "generic_anchor_token_share_band", "specific_family_band"],
    kind="stable",
)

additional_group_results = {}
for group_column in ADDITIONAL_GROUP_COLUMNS:
    grouped_metrics = per_query_metrics_df.merge(
        query_diagnostics_df[["case_id", group_column]],
        on="case_id",
        how="left",
        validate="many_to_one",
    )
    additional_group_results[group_column] = aggregate_metrics(
        grouped_metrics,
        ["query_method", "method_key", "retrieval_method", group_column],
    ).sort_values(["retrieval_method", group_column], kind="stable")

component_columns = {
    "dense_global_review": "dense_component_present",
    "bm25_global_review": "bm25_component_present",
    "metadata_functional_graph": "metadata_graph_component_present",
    "historical_review_graph": "review_reputation_graph_component_present",
}
component_coverage_df = pd.DataFrame([
    {
        "component": component,
        "query_count": int(query_diagnostics_df[column].sum()),
        "query_coverage_rate": float(query_diagnostics_df[column].mean()),
        "total_queries": int(len(query_diagnostics_df)),
    }
    for component, column in component_columns.items()
])

method_runtime_sec = {
    "dense_bert": component_runtime["query_embedding"] + component_runtime["dense_global_review_faiss"],
    "bm25": component_runtime["bm25_global_review_retrieval"],
    "hybrid_dense_bm25": (
        component_runtime["query_embedding"]
        + component_runtime["dense_global_review_faiss"]
        + component_runtime["bm25_global_review_retrieval"]
        + component_runtime["dense_bm25_global_review_rrf"]
    ),
    "graph_hybrid": (
        component_runtime["query_embedding"]
        + component_runtime["dense_global_review_faiss"]
        + component_runtime["bm25_global_review_retrieval"]
        + component_runtime["metadata_functional_graph_retrieval"]
        + component_runtime["review_reputation_graph_retrieval"]
        + component_runtime["graph_hybrid_global_review_rrf"]
    ),
}
runtime_df = pd.DataFrame([
    {
        "query_method": QUERY_METHOD_LABEL,
        "method_key": method_key,
        "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
        "candidate_pool_depth": MAX_RETRIEVAL_K,
        "n_queries": int(len(evaluation_queries_df)),
        "online_runtime_sec": float(method_runtime_sec[method_key]),
        "mean_online_runtime_ms_per_query": float(
            1000.0 * method_runtime_sec[method_key] / len(evaluation_queries_df)
        ),
        "runtime_scope": "component-summed online top-1000 generation; excludes offline indexes and parquet export",
    }
    for method_key in METHOD_KEYS
])
runtime_components_df = pd.DataFrame([
    {"component": component, "runtime_sec": float(seconds)}
    for component, seconds in component_runtime.items()
] + [
    {"component": "offline_item_and_graph_index_build", "runtime_sec": float(offline_index_runtime_sec)},
    {"component": "candidate_parquet_export", "runtime_sec": float(sum(candidate_write_runtime_sec.values()))},
])

selection_metric = f"HitRate@{SELECTION_K}"
selection_ndcg = f"NDCG@{SELECTION_K}"
selection_mrr = f"MRR@{SELECTION_K}"
regime_consistency_df = (
    results_by_regime_df
    .groupby(["method_key", "retrieval_method"], observed=True)[selection_metric]
    .agg(
        regime_hit_rate_min="min",
        regime_hit_rate_max="max",
        regime_hit_rate_mean="mean",
        regime_hit_rate_std=lambda values: float(values.std(ddof=0)),
    )
    .reset_index()
)
method_selection_df = (
    results_overall_df
    .merge(regime_consistency_df, on=["method_key", "retrieval_method"], how="left", validate="one_to_one")
    .merge(
        runtime_df[["method_key", "online_runtime_sec", "mean_online_runtime_ms_per_query"]],
        on="method_key",
        how="left",
        validate="one_to_one",
    )
)
method_selection_df["method_order_index"] = method_selection_df["method_key"].map(
    {method_key: index for index, method_key in enumerate(METHOD_KEYS)}
)
method_selection_df = method_selection_df.sort_values(
    [
        selection_metric,
        selection_ndcg,
        selection_mrr,
        "regime_hit_rate_min",
        "regime_hit_rate_std",
        "online_runtime_sec",
        "method_order_index",
    ],
    ascending=[False, False, False, False, True, True, True],
    kind="stable",
).reset_index(drop=True)
method_selection_df.insert(0, "selection_rank", np.arange(1, len(method_selection_df) + 1))
method_selection_df["primary_stage1_metric"] = selection_metric
method_selection_df["selection_rule"] = (
    f"maximize {selection_metric}; then {selection_ndcg}, {selection_mrr}, "
    "minimum regime HitRate, regime HitRate stability, and online runtime"
)
automatic_winner_method_key = str(method_selection_df.iloc[0]["method_key"])
if WINNER_METHOD_OVERRIDE is None:
    WINNER_METHOD_KEY = automatic_winner_method_key
    WINNER_SELECTION_SOURCE = "automatic_selection_rank_1"
else:
    WINNER_METHOD_KEY = str(WINNER_METHOD_OVERRIDE).strip()
    if WINNER_METHOD_KEY not in METHOD_KEYS:
        raise RuntimeError(
            f"WINNER_METHOD_OVERRIDE must be one of {METHOD_KEYS}, found {WINNER_METHOD_KEY!r}."
        )
    WINNER_SELECTION_SOURCE = "config_override"

WINNER_METHOD_LABEL = METHOD_DISPLAY_NAMES[WINNER_METHOD_KEY]
WINNER_CANDIDATE_PATH = METHOD_CANDIDATE_PATHS[WINNER_METHOD_KEY]
method_selection_df["is_selected_winner"] = method_selection_df["method_key"].eq(WINNER_METHOD_KEY)
method_selection_df["selection_status"] = np.where(
    method_selection_df["is_selected_winner"],
    "selected_for_downstream",
    "not_selected",
)
method_selection_df["winner_selection_source"] = WINNER_SELECTION_SOURCE
method_selection_df["generic_winner_alias_written"] = False
method_selection_df = method_selection_df.drop(columns=["method_order_index"])

optional_export_columns = [
    column for column in OPTIONAL_QUERY_AUDIT_COLUMNS
    if column in evaluation_queries_df.columns
]
query_export_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "target_selection_mode",
    "target_timestamp_ms",
    ACTIVE_QUERY_COLUMN,
    "active_query_text",
    "query_text_retrieval",
    "query_text_graph",
    "query_method",
    *QUERY_STRUCTURE_COLUMNS,
    *QUERY_PASSTHROUGH_COLUMNS,
    *optional_export_columns,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    query_export_columns.append(COMPATIBILITY_QUERY_ALIAS)
query_export_columns = list(dict.fromkeys(query_export_columns))
evaluation_queries_export_df = evaluation_queries_df[query_export_columns].copy()

facet_filter_audit_record = {
    str(key): value.item() if isinstance(value, np.generic) else value
    for key, value in facet_filter_audit_df.iloc[0].to_dict().items()
}
automatic_top_method = str(method_selection_df.iloc[0]["retrieval_method"])
winner_selection_row = method_selection_df.loc[
    method_selection_df["method_key"].eq(WINNER_METHOD_KEY)
].iloc[0]
effective_candidate_k = min(int(MAX_RETRIEVAL_K), int(catalog_size))

run_manifest = {
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "evidence_scope": EVIDENCE_SCOPE,
    "query_evidence_scope": "target_review_safe_signals_only",
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_graph_enabled": True,
    "brand_in_functional_graph": BRAND_IN_FUNCTIONAL_GRAPH,
    "brand_in_retrieval_text": True,
    "brand_in_candidate_output": True,
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "user_prior_enabled": False,
    "target_metadata_query_fallback_enabled": False,
    "target_metadata_loaded_for_scoring": False,
    "raw_review_text_loaded": False,
    "historical_review_data_loaded": False,
    "historical_review_facet_values_loaded": True,
    "user_prior_data_loaded": False,
    "item_facet_value_load_filter": "metadata is_core_graph_facet plus brand is_brand_graph_facet plus review is_global_review_graph_facet",
    "dense_source": DENSE_TEXT_COLUMN,
    "sparse_source": SPARSE_TEXT_COLUMN,
    "graph_source": "metadata functional facets plus preserved brand edges plus historical review-derived facets; query scoring uses metadata and review channels only",
    "retrieval_methods": [METHOD_DISPLAY_NAMES[key] for key in METHOD_KEYS],
    "active_query_column": ACTIVE_QUERY_COLUMN,
    "query_contract_path": str(QUERY_CONTRACT_PATH),
    "retrieval_artifact_manifest_path": str(RETRIEVAL_ARTIFACT_MANIFEST_PATH),
    "candidate_pool_depth": MAX_RETRIEVAL_K,
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": MAX_RETRIEVAL_K,
    "expected_candidate_count_per_query": int(EXPECTED_CANDIDATE_K),
    "catalog_size": int(catalog_size),
    "bm25_positive_only_filter": False,
    "bm25_nonpositive_candidates_allowed": True,
    "bm25_tie_break_policy": "stable_catalog_item_id_order",
    "exact_k_validation_passed": True,
    "evaluation_ks": EVAL_KS,
    "fixed_downstream_reranking_depth": SELECTION_K,
    "method_selection_primary_metric": selection_metric,
    "method_selection_secondary_evidence": [
        selection_ndcg,
        selection_mrr,
        "regime_hit_rate_min",
        "regime_hit_rate_std",
        "online_runtime_sec",
    ],
    "automatic_top_method_key": automatic_winner_method_key,
    "automatic_top_method_label": automatic_top_method,
    "winner_method_key": WINNER_METHOD_KEY,
    "winner_method_label": WINNER_METHOD_LABEL,
    "winner_candidate_path": str(WINNER_CANDIDATE_PATH),
    "winner_selection_source": WINNER_SELECTION_SOURCE,
    "winner_contract_path": str(WINNER_MANIFEST_PATH),
    "winner_contract_written": True,
    "generic_downstream_winner_alias_written": False,
    "rrf_k": RRF_K,
    "hybrid_weights": HYBRID_WEIGHTS,
    "graph_hybrid_weights": GRAPH_HYBRID_WEIGHTS,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "bm25_zero_score_candidates_included": True,
    "graph_phrase_first_matching": True,
    "graph_phrase_bonus": GRAPH_PHRASE_BONUS,
    "run_smoke_test": RUN_SMOKE_TEST,
    "expected_query_rows": EXPECTED_ROWS,
    "evaluated_query_rows": int(len(evaluation_queries_df)),
    "expected_per_query_metric_rows": int(len(evaluation_queries_df) * len(METHOD_KEYS)),
    "actual_per_query_metric_rows": int(len(per_query_metrics_df)),
    "expected_regime_counts": EXPECTED_REGIME_COUNTS,
    "item_count": int(len(item_docs)),
    "metadata_functional_facet_rows": int(len(core_facets)),
    "brand_facet_rows": int(len(brand_facets)),
    "historical_review_facet_rows": int(len(review_facets)),
    "metadata_graph_edges": int(
        graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum()
    ),
    "brand_graph_edges": int(
        graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum()
    ),
    "review_reputation_graph_edges": int(
        graph_edges["is_review_derived"].fillna(False).astype(bool).sum()
    ),
    "global_review_graph_edges": int(len(graph_edges)),
    "production_graph_validation": {
        "review_derived_rows_used": int(facet_filter_audit_df["review_derived_rows_used"].iloc[0]),
        "brand_rows_used": int(facet_filter_audit_df["brand_rows_used"].iloc[0]),
        "generic_anchor_rows_used": int(facet_filter_audit_df["generic_anchor_rows_used"].iloc[0]),
        "generic_utility_rows_used": int(facet_filter_audit_df["generic_utility_rows_used"].iloc[0]),
        "context_utility_rows_used": int(facet_filter_audit_df["context_utility_rows_used"].iloc[0]),
        "disallowed_source_rows_used": int(facet_filter_audit_df["disallowed_source_rows_used"].iloc[0]),
    },
    "facet_filter_audit": facet_filter_audit_record,
    "component_query_coverage": {
        str(row.component): {
            "query_count": int(row.query_count),
            "query_coverage_rate": float(row.query_coverage_rate),
        }
        for row in component_coverage_df.itertuples(index=False)
    },
    "candidate_paths": {
        method_key: str(METHOD_CANDIDATE_PATHS[method_key])
        for method_key in METHOD_KEYS
    },
    "candidate_row_counts": {
        method_key: int(candidate_row_counts[method_key])
        for method_key in METHOD_KEYS
    },
    "candidate_target_hit_counts": {
        method_key: int(candidate_target_hit_counts[method_key])
        for method_key in METHOD_KEYS
    },
    "runtime_sec": {
        "load_and_validation": float(load_validation_runtime_sec),
        "offline_indexes": float(offline_index_runtime_sec),
        "evaluation_loop": float(evaluation_loop_runtime_sec),
        "candidate_export": float(sum(candidate_write_runtime_sec.values())),
    },
    "output_paths": {
        "overall_results": str(RESULTS_OVERALL_PATH),
        "results_by_pool_depth": str(RESULTS_BY_POOL_DEPTH_PATH),
        "results_by_regime": str(RESULTS_BY_REGIME_PATH),
        "results_by_regime_pool_depth": str(RESULTS_BY_REGIME_POOL_DEPTH_PATH),
        "per_query_metrics": str(PER_QUERY_METRICS_PATH),
        "runtime_summary": str(RUNTIME_PATH),
        "method_selection": str(METHOD_SELECTION_PATH),
        "run_manifest": str(RUN_MANIFEST_PATH),
        "winner_contract": str(WINNER_MANIFEST_PATH),
    },
}

winner_manifest = {
    "contract_version": WINNER_CONTRACT_VERSION,
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "stage": "stage1_query_only_retrieval",
    "selection_source": WINNER_SELECTION_SOURCE,
    "automatic_winner_method_key": automatic_winner_method_key,
    "winner_method_key": WINNER_METHOD_KEY,
    "winner_method_label": WINNER_METHOD_LABEL,
    "winner_candidate_path": str(WINNER_CANDIDATE_PATH),
    "winner_candidate_row_count": int(candidate_row_counts[WINNER_METHOD_KEY]),
    "winner_target_hit_count": int(candidate_target_hit_counts[WINNER_METHOD_KEY]),
    "selection_rank": int(winner_selection_row["selection_rank"]),
    "selection_metric": selection_metric,
    "selection_metrics": {
        selection_metric: float(winner_selection_row[selection_metric]),
        selection_ndcg: float(winner_selection_row[selection_ndcg]),
        selection_mrr: float(winner_selection_row[selection_mrr]),
        "regime_hit_rate_min": float(winner_selection_row["regime_hit_rate_min"]),
        "regime_hit_rate_std": float(winner_selection_row["regime_hit_rate_std"]),
        "online_runtime_sec": float(winner_selection_row["online_runtime_sec"]),
    },
    "selection_rule": str(winner_selection_row["selection_rule"]),
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": int(MAX_RETRIEVAL_K),
    "effective_candidate_count_per_query": int(effective_candidate_k),
    "catalog_size": int(catalog_size),
    "query_count": int(len(evaluation_queries_df)),
    "exact_k_validation_passed": True,
    "candidate_schema_columns": CANDIDATE_COLUMNS,
    "query_evidence_scope": "target_review_safe_signals_only",
    "retrieval_evidence_scope": EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_graph_enabled": True,
    "brand_in_functional_graph": BRAND_IN_FUNCTIONAL_GRAPH,
    "brand_in_retrieval_text": True,
    "brand_in_candidate_output": True,
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "user_prior_enabled": False,
    "method_selection_path": str(METHOD_SELECTION_PATH),
    "stage1_run_manifest_path": str(RUN_MANIFEST_PATH),
    "evaluation_queries_path": str(EVALUATION_QUERIES_PATH),
}

results_overall_df.to_csv(RESULTS_OVERALL_PATH, index=False, encoding="utf-8-sig")
results_by_pool_depth_df.to_csv(RESULTS_BY_POOL_DEPTH_PATH, index=False, encoding="utf-8-sig")
results_by_regime_df.to_csv(RESULTS_BY_REGIME_PATH, index=False, encoding="utf-8-sig")
results_by_regime_pool_depth_df.to_csv(
    RESULTS_BY_REGIME_POOL_DEPTH_PATH,
    index=False,
    encoding="utf-8-sig",
)
results_by_query_structure_df.to_csv(
    RESULTS_BY_QUERY_STRUCTURE_PATH,
    index=False,
    encoding="utf-8-sig",
)
for group_column, result_frame in additional_group_results.items():
    result_frame.to_csv(
        ADDITIONAL_GROUP_OUTPUT_PATHS[group_column],
        index=False,
        encoding="utf-8-sig",
    )
per_query_metrics_df.to_parquet(PER_QUERY_METRICS_PATH, index=False)
query_diagnostics_df.to_parquet(QUERY_DIAGNOSTICS_PATH, index=False)
evaluation_queries_export_df.to_parquet(EVALUATION_QUERIES_PATH, index=False)
runtime_df.to_csv(RUNTIME_PATH, index=False, encoding="utf-8-sig")
runtime_components_df.to_csv(RUNTIME_COMPONENTS_PATH, index=False, encoding="utf-8-sig")
component_coverage_df.to_csv(COMPONENT_COVERAGE_PATH, index=False, encoding="utf-8-sig")
facet_filter_audit_df.to_csv(FACET_FILTER_AUDIT_PATH, index=False, encoding="utf-8-sig")
method_selection_df.to_csv(METHOD_SELECTION_PATH, index=False, encoding="utf-8-sig")
with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, ensure_ascii=False, indent=2)
with open(WINNER_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(winner_manifest, file, ensure_ascii=False, indent=2)

print("Output:", RESULTS_OVERALL_PATH)
print("Output:", PER_QUERY_METRICS_PATH)
print("Output:", METHOD_SELECTION_PATH)
print("Output:", RUN_MANIFEST_PATH)
print("Output:", WINNER_MANIFEST_PATH)
print("Selected query-only winner:", WINNER_METHOD_KEY, "-", WINNER_METHOD_LABEL)

In [ ]:
# =========================================================
# Final Validation
# =========================================================
expected_evaluation_rows = SMOKE_TEST_N if RUN_SMOKE_TEST else EXPECTED_ROWS
expected_metric_rows = expected_evaluation_rows * len(METHOD_KEYS)
effective_candidate_k = min(int(MAX_RETRIEVAL_K), int(catalog_size))
expected_candidate_rows = expected_evaluation_rows * effective_candidate_k
expected_case_ids = set(evaluation_queries_df["case_id"].astype(str))

if METHOD_KEYS != ["dense_bert", "bm25", "hybrid_dense_bm25", "graph_hybrid"]:
    raise RuntimeError("The production method loop must contain exactly four Global Review methods.")
if len(per_query_metrics_df) != expected_metric_rows:
    raise RuntimeError(
        f"Per-query metric row count mismatch: {len(per_query_metrics_df)} vs {expected_metric_rows}"
    )
if len(results_overall_df) != len(METHOD_KEYS):
    raise RuntimeError("Overall results must contain exactly one row per retrieval method.")
for method_key in METHOD_KEYS:
    observed = int(per_query_metrics_df["method_key"].eq(method_key).sum())
    if observed != expected_evaluation_rows:
        raise RuntimeError(f"{method_key} evaluated {observed} queries; expected {expected_evaluation_rows}.")

required_candidate_columns = set(CANDIDATE_COLUMNS)
method_case_sets = {}
for method_key, candidate_path in METHOD_CANDIDATE_PATHS.items():
    candidate_parquet = pq.ParquetFile(candidate_path)
    if candidate_parquet.metadata.num_rows != candidate_row_counts[method_key]:
        raise RuntimeError(f"Candidate parquet row count mismatch for {method_key}.")
    missing_columns = sorted(required_candidate_columns.difference(candidate_parquet.schema.names))
    if missing_columns:
        raise RuntimeError(f"Candidate cache missing columns for {method_key}: {missing_columns}")
    if candidate_row_counts[method_key] != expected_candidate_rows:
        raise RuntimeError(
            f"Candidate row count mismatch for {method_key}: {candidate_row_counts[method_key]} "
            f"vs expected {expected_candidate_rows}."
        )
    candidate_check = pd.read_parquet(
        candidate_path,
        columns=["case_id", "method_key", "retrieval_method", "candidate_parent_asin", "candidate_brand_facet_text", "candidate_rank", "candidate_score"],
    )
    candidate_check["case_id"] = candidate_check["case_id"].astype(str)
    method_case_sets[method_key] = set(candidate_check["case_id"])
    if method_case_sets[method_key] != expected_case_ids:
        raise RuntimeError(f"{method_key} candidate cases do not match active query cases.")
    if not candidate_check["method_key"].astype(str).eq(method_key).all():
        raise RuntimeError(f"{method_key} candidate cache has an unexpected method_key value.")
    if not candidate_check["retrieval_method"].astype(str).eq(METHOD_DISPLAY_NAMES[method_key]).all():
        raise RuntimeError(f"{method_key} candidate cache has an unexpected retrieval_method label.")
    if candidate_check.duplicated(["case_id", "candidate_parent_asin"]).any():
        raise RuntimeError(f"{method_key} candidate cache contains duplicate items within a query.")
    if not candidate_check["candidate_parent_asin"].astype(str).isin(item_id_set).all():
        raise RuntimeError(f"{method_key} candidate cache contains an item outside the sorted catalog.")

    expected_brand = candidate_check["candidate_parent_asin"].astype(str).map(
        item_docs.set_index(ITEM_ID_COLUMN)[BRAND_TEXT_COLUMN]
    ).fillna("").astype(str)
    if not candidate_check["candidate_brand_facet_text"].fillna("").astype(str).eq(expected_brand).all():
        raise RuntimeError(f"{method_key} candidate brand values do not match the Notebook 04 catalog.")
    scores = pd.to_numeric(candidate_check["candidate_score"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(scores).all():
        raise RuntimeError(f"{method_key} candidate scores contain non-finite values.")
    count_by_case = candidate_check.groupby("case_id", sort=False).size()
    if not count_by_case.eq(effective_candidate_k).all():
        bad_counts = count_by_case.loc[~count_by_case.eq(effective_candidate_k)].head(10).to_dict()
        raise RuntimeError(f"{method_key} violates exact-K candidate counts: {bad_counts}")
    for case_id, group in candidate_check.groupby("case_id", sort=False):
        ranks = group.sort_values("candidate_rank", kind="stable")["candidate_rank"].astype(int).tolist()
        if ranks != list(range(1, effective_candidate_k + 1)):
            raise RuntimeError(f"{method_key} ranks are not exactly 1..{effective_candidate_k} for {case_id}.")

if any(case_set != expected_case_ids for case_set in method_case_sets.values()):
    raise RuntimeError("Candidate query coverage differs across Stage 1 methods.")

if len(runtime_df) != len(METHOD_KEYS) or set(runtime_df["method_key"]) != set(METHOD_KEYS):
    raise RuntimeError("Runtime summary must contain exactly the four Global Review methods.")
if len(method_selection_df) != len(METHOD_KEYS):
    raise RuntimeError("Method-selection table must contain exactly four methods.")
if method_selection_df["selection_rank"].tolist() != list(range(1, len(METHOD_KEYS) + 1)):
    raise RuntimeError("Method-selection ranks must be contiguous from 1.")
if int(method_selection_df["is_selected_winner"].sum()) != 1:
    raise RuntimeError("Method-selection table must contain exactly one selected winner.")
selected_method_key = str(
    method_selection_df.loc[method_selection_df["is_selected_winner"], "method_key"].iloc[0]
)
if selected_method_key != WINNER_METHOD_KEY:
    raise RuntimeError("Selected winner in the method table does not match the winner contract variables.")
if method_selection_df["generic_winner_alias_written"].any():
    raise RuntimeError("Notebook 07 must not duplicate the winner candidate cache under a generic alias.")
if not method_selection_df["primary_stage1_metric"].eq(f"HitRate@{SELECTION_K}").all():
    raise RuntimeError("HitRate at the fixed downstream depth must be the primary selection criterion.")

if core_facets.empty or brand_facets.empty or review_facets.empty or graph_edges.empty:
    raise RuntimeError(
        "Metadata facets, brand facets, historical review-derived facets, and Global Review graph edges must be non-empty."
    )
if int(facet_filter_audit_df["historical_facet_values_loaded"].iloc[0]) <= 0:
    raise RuntimeError("Historical review-derived facet values must be loaded.")
if int(facet_filter_audit_df["review_derived_rows_used"].iloc[0]) <= 0:
    raise RuntimeError("Historical review-derived graph rows must be greater than zero.")
if int(facet_filter_audit_df["brand_rows_used"].iloc[0]) <= 0:
    raise RuntimeError("Brand facet rows must be preserved in the Global Review catalog graph.")
for column in [
    "generic_anchor_rows_used",
    "generic_utility_rows_used",
    "context_utility_rows_used",
    "disallowed_source_rows_used",
    "core_mask_mismatch_rows",
    "brand_mask_mismatch_rows",
    "global_mask_mismatch_rows",
]:
    if int(facet_filter_audit_df[column].iloc[0]) != 0:
        raise RuntimeError(f"Global Review facet filter audit failed: {column}")
if bool(facet_filter_audit_df["brand_query_matching_enabled"].iloc[0]):
    raise RuntimeError("Synthetic-query brand matching must remain disabled.")

component_coverage = component_coverage_df.set_index("component")
if float(component_coverage.loc["dense_global_review", "query_coverage_rate"]) != 1.0:
    raise RuntimeError("Dense retrieval must contribute candidates for every query.")
for component in ["bm25_global_review", "metadata_functional_graph", "historical_review_graph"]:
    if int(component_coverage.loc[component, "query_count"]) <= 0:
        message = f"{component} contributed to zero evaluated queries."
        if RUN_SMOKE_TEST:
            print("WARNING:", message)
        else:
            raise RuntimeError(message)

if not evaluation_queries_df["query_text_retrieval"].eq(evaluation_queries_df["active_query_text"]).all():
    raise RuntimeError("Dense and BM25 did not use the exact active query.")
if ACTIVE_QUERY_COLUMN == CLEAN_QUERY_COLUMN:
    raise RuntimeError("query_clean cannot be the active production query.")

required_manifest_values = {
    "evidence_scope": EVIDENCE_SCOPE,
    "query_evidence_scope": "target_review_safe_signals_only",
    "historical_review_reputation_enabled": True,
    "review_reputation_graph_enabled": True,
    "brand_graph_enabled": True,
    "brand_in_functional_graph": BRAND_IN_FUNCTIONAL_GRAPH,
    "brand_in_retrieval_text": True,
    "brand_in_candidate_output": True,
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "user_prior_enabled": False,
    "target_metadata_query_fallback_enabled": False,
    "historical_review_data_loaded": False,
    "historical_review_facet_values_loaded": True,
    "user_prior_data_loaded": False,
    "generic_downstream_winner_alias_written": False,
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": MAX_RETRIEVAL_K,
    "catalog_size": int(catalog_size),
    "bm25_positive_only_filter": False,
    "bm25_nonpositive_candidates_allowed": True,
    "bm25_tie_break_policy": "stable_catalog_item_id_order",
    "exact_k_validation_passed": True,
    "winner_method_key": WINNER_METHOD_KEY,
    "winner_method_label": WINNER_METHOD_LABEL,
    "winner_candidate_path": str(WINNER_CANDIDATE_PATH),
    "winner_contract_path": str(WINNER_MANIFEST_PATH),
    "winner_contract_written": True,
}
manifest_mismatches = {
    key: run_manifest.get(key)
    for key, expected_value in required_manifest_values.items()
    if run_manifest.get(key) != expected_value
}
if manifest_mismatches:
    raise RuntimeError(f"Stage 1 run manifest mismatch: {manifest_mismatches}")
if run_manifest.get("retrieval_methods") != [METHOD_DISPLAY_NAMES[key] for key in METHOD_KEYS]:
    raise RuntimeError("Stage 1 run manifest retrieval_methods mismatch.")

winner_contract_check = load_json(WINNER_MANIFEST_PATH)
required_winner_values = {
    "contract_version": WINNER_CONTRACT_VERSION,
    "category_id": CATEGORY_ID,
    "winner_method_key": WINNER_METHOD_KEY,
    "winner_method_label": WINNER_METHOD_LABEL,
    "winner_candidate_path": str(WINNER_CANDIDATE_PATH),
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": int(MAX_RETRIEVAL_K),
    "effective_candidate_count_per_query": int(effective_candidate_k),
    "catalog_size": int(catalog_size),
    "query_count": int(expected_evaluation_rows),
    "exact_k_validation_passed": True,
    "query_evidence_scope": "target_review_safe_signals_only",
    "retrieval_evidence_scope": EVIDENCE_SCOPE,
    "brand_graph_enabled": True,
    "brand_in_functional_graph": BRAND_IN_FUNCTIONAL_GRAPH,
    "brand_in_retrieval_text": True,
    "brand_in_candidate_output": True,
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "user_prior_enabled": False,
}
winner_mismatches = {
    key: winner_contract_check.get(key)
    for key, expected_value in required_winner_values.items()
    if winner_contract_check.get(key) != expected_value
}
if winner_mismatches:
    raise RuntimeError(f"Query-only winner contract mismatch: {winner_mismatches}")
if not WINNER_CANDIDATE_PATH.exists():
    raise RuntimeError("Selected winner candidate cache does not exist.")

required_outputs = [
    RESULTS_OVERALL_PATH,
    RESULTS_BY_POOL_DEPTH_PATH,
    RESULTS_BY_REGIME_PATH,
    RESULTS_BY_REGIME_POOL_DEPTH_PATH,
    RESULTS_BY_QUERY_STRUCTURE_PATH,
    PER_QUERY_METRICS_PATH,
    QUERY_DIAGNOSTICS_PATH,
    EVALUATION_QUERIES_PATH,
    RUNTIME_PATH,
    RUNTIME_COMPONENTS_PATH,
    COMPONENT_COVERAGE_PATH,
    FACET_FILTER_AUDIT_PATH,
    METHOD_SELECTION_PATH,
    RUN_MANIFEST_PATH,
    WINNER_MANIFEST_PATH,
    *METHOD_CANDIDATE_PATHS.values(),
    *ADDITIONAL_GROUP_OUTPUT_PATHS.values(),
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Missing outputs: {missing_outputs}")

print("Rows: queries", expected_evaluation_rows)
print("Rows: per-query metrics", len(per_query_metrics_df))
print("Rows: candidate caches", candidate_row_counts)
print("Selected query-only winner:", WINNER_METHOD_KEY, "-", WINNER_METHOD_LABEL)
print("Validation: PASS")
